# Computing ED2 and EDGE2 scores

This notebook demonstrates how to iterate through a distribution of trees, compute ED2 and EDGE2 scores, and write out the distributions of scores.

We will build up gradually to the full computation. We will begin by using a small tree of birds; then try the full tree. Then we will loop over the birds tree, and finally loop over the full tree - which can take a while, so we want to be sure everything works first.

## Loading a tree

First, check you can import the necessary Python modules.

In [ ]:
import numpy as np
import datetime

import tree_loading
import tree_labelling
import tree_dating
import tree_metrics

We are going to load a tree and annotate it with the taxonomy. By default we will look for the taxonomy file `ott3.7.3/taxonomy.tsv`. We will load a tree from the distribution of trees available at .... Here we put the trees in a folder `trees/`, and we will load tree 407, which is the tree in the topological variation distribution that has the median PD.

In [ ]:
# Load taxonomic info
taxa = tree_loading.load_metadata(date_cache=None, annotations=None)

In [ ]:
# Load and annotate tree number 407, the "median" tree
tree = tree_loading.build_and_annotate_tree(phylogeny_nodes=None, 
                                            taxa=taxa,
                                            tree_filename="trees/dated_tree_topo_sample_407.tre",
                                            has_branch_lengths=True,
                                            suppress_logging=True)

Lastly, we add some internal labels indicating taxonomic positioning.

In [ ]:
tree_labelling.add_anc_ranks(tree)

## Computing EDGE2

Now we assign IUCN Red List status to each species, where available. By default we use `config/latest_iucn_2025.csv` (provided by Jialiang Guo).

In [ ]:
tree_metrics.assign_iucn_status(tree)

Then we convert statuses to probabilities using the curve from the EDGE2 protocol (Gumbs et al. 2024), which by default is found in `config/p_extinction.csv`. We sample from the relevant part of the curve for each status; to use the median instead, set `randomise_risk=False`. Where no status is available we randomly sample an extinction probability from the curve.

![Status -> extinction probability curve](examples/iucn_curve.png)

In [ ]:
rng = np.random.default_rng(seed=1)
tree_metrics.assign_extinction_risks(tree, rng=rng, randomise_risk=True)

Finally, we compute ED2 and EDGE2 scores. This will annotate all the leaves of the tree with the properties `ed2_score` and `edge2_score`.

In [ ]:
tree_metrics.compute_edge2_scores(tree)

# for example, print the scores for Homo sapiens
for n in tree.search_nodes(name="Homo_sapiens_ott770315"):
    print(n.name, n.props["iucn_status"], n.props["ed2_score"], n.props["edge2_score"])
    break

## A distribution of EDGE2 scores for a small tree

Now we have got the basic code working, we can produce a distribution of scores. First we will loop over a single birds tree. (The extinction probabilities are sampled from a curve, so we will still get a distribution of scores even with only one tree.)

The output file will give for each species the mean, median, min, max and 95% confidence interval for EDGE2, ED2 and extinction probability.

In [ ]:
rng = np.random.default_rng(seed=1)

scores_dict = {} # store results here
num_trees = 1
num_samples = 1000

for t in range(num_trees):
    tree = tree_loading.build_and_annotate_tree(phylogeny_nodes=None, 
                                                taxa=taxa,
                                                tree_filename="config/birds.tre",
                                                has_branch_lengths=True,
                                                suppress_logging=True)
    tree_labelling.add_anc_ranks(tree)
    tree_metrics.assign_iucn_status(tree)

    print("Sampling", num_samples, "extinction probabilities...")
    for s in range(num_samples):
        if s+1 == num_samples:
            print("Complete.")
        elif (s+1) % 100 == 0:
            print("Sampled", s+1, "...")
        tree_metrics.assign_extinction_risks(tree, rng=rng, randomise_risk=True)
        tree_metrics.compute_edge2_scores(tree, scores_dict=scores_dict)

print("Writing out scores...")
tree_metrics.write_edge2_scores("edge2_scores_birds.csv", scores_dict)
print("Done.")

## A distribution of EDGE2 scores for all extant described species

Finally, let's try looping over some complete trees, and for each tree sample multiple extinction probabilities.

We will limit the output file to the top 100 species per phylum, by mean EDGE2 score; otherwise the output becomes a bit unmanageable.

Let's start with 5 trees, with extinction probabilites sampled 5 times for each. On my machine this takes about 7 minutes.

In [ ]:
rng = np.random.default_rng(seed=1)

scores_dict = {} # store results here

num_trees = 5
num_samples = 5

start_time = datetime.datetime.now()
for t in range(num_trees):
    print("Tree number",
          t+1,
          "/ projected end time:",
          start_time + num_trees*(datetime.datetime.now() - start_time)/t if t > 0 else "first iteration, no estimate yet")
    
    tree = tree_loading.build_and_annotate_tree(phylogeny_nodes=None, 
                                                taxa=taxa,
                                                tree_filename="trees/dated_tree_topo_sample_{:d}.tre".format(itr+1),
                                                has_branch_lengths=True,
                                                suppress_logging=True)
    tree_labelling.add_anc_ranks(tree)
    tree_metrics.assign_iucn_status(tree)

    for s in range(num_samples):
        tree_metrics.assign_extinction_risks(tree, rng=rng, randomise_risk=True)
        tree_metrics.compute_edge2_scores(tree, scores_dict=scores_dict)

print("Writing out scores...")
tree_metrics.write_edge2_scores("edge2_scores_top_100s.csv", scores_dict, per_phylum=100)
print("Done.")